# Limpieza de Datos - Taco Bueno Pricing

Este notebook limpia los datos de la tabla `workspace.default.taco_bueno_raw_data` (85M+ filas, 597 CSVs).

**Pasos:**
1. Cargar datos y revisar schema
2. Perfil de calidad (nulos, vacíos, duplicados)
3. Conversión de tipos (fechas, números)
4. Estandarización de categorías
5. Guardar tabla limpia

**Correcciones aplicadas:**
- `spark.sql.ansi.enabled = false` para que CAST devuelva NULL en vez de fallar
- Formato de fecha `M/d/yyyy` (no `MM/dd/yyyy`) para manejar meses/días de 1 dígito
- CTAS en vez de `df.write.saveAsTable` para mejor rendimiento en serverless

In [0]:
# Configuracion: desactivar ANSI mode para que CAST devuelva NULL en valores invalidos
spark.conf.set("spark.sql.ansi.enabled", "false")

# Cargar la tabla raw
table_name = "workspace.default.taco_bueno_raw_data"
df = spark.table(table_name)

print(f"Filas: {df.count():,}")
print(f"Columnas: {len(df.columns)}")
df.printSchema()

In [0]:
from pyspark.sql.functions import col, count, when, isnan, isnull, trim

# Contar nulos y strings vacíos por columna (una sola pasada + count optimizado)
total_count = df.count()  # Calcular una sola vez, no por cada columna

print("=== Perfil de calidad: nulos y vacíos ===")
null_counts = df.select([
    count(when(col(c).isNull() | (trim(col(c)) == ""), c)).alias(c)
    for c in df.columns
]).collect()[0]

for c in df.columns:
    null_val = null_counts[c]
    pct = (null_val / total_count) * 100
    status = "⚠️" if null_val > 0 else "✅"
    print(f"  {status} {c}: {null_val:,} nulos/vacíos ({pct:.2f}%)")

In [0]:
# Verificar filas duplicadas
total_rows = df.count()
distinct_rows = df.distinct().count()
duplicates = total_rows - distinct_rows

print(f"=== Duplicados ===")
print(f"  Filas totales: {total_rows:,}")
print(f"  Filas únicas: {distinct_rows:,}")
print(f"  Duplicados: {duplicates:,} ({(duplicates/total_rows)*100:.2f}%)")

In [0]:
from pyspark.sql.functions import to_date, regexp_replace, col as c
from pyspark.sql.types import DoubleType, IntegerType

# Limpiar y convertir tipos
df_clean = df.select(
    # Store → entero
    c("Store").cast("int").alias("store_id"),
    
    # Date → fecha (formato M/d/yyyy maneja 1 y 2 dígitos)
    to_date(col("Date"), "M/d/yyyy").alias("date"),
    
    # Columnas numéricas (limpiar comas si las hay)
    regexp_replace(c("Total"), ",", "").cast(DoubleType()).alias("total"),
    c("Check_Number").alias("check_number"),
    c("Time_Card").alias("time_card"),
    c("Begin_Time").alias("begin_time"),
    c("End_Time").alias("end_time"),
    regexp_replace(c("Total_Discount"), ",", "").cast(DoubleType()).alias("total_discount"),
    regexp_replace(c("Order_Voided"), ",", "").cast(DoubleType()).alias("order_voided"),
    regexp_replace(c("Tax"), ",", "").cast(DoubleType()).alias("tax"),
    c("Quantity").cast(IntegerType()).alias("quantity"),
    trim(c("Transaction_Type")).alias("transaction_type"),
    trim(c("Menu_Item")).alias("menu_item"),
    c("Menu_Item_Ext_Ref").alias("menu_item_ext_ref"),
    trim(c("Pop")).alias("pop"),
    c("POS_Code").alias("pos_code"),
    regexp_replace(c("Menu_Price"), ",", "").cast(DoubleType()).alias("menu_price"),
    regexp_replace(c("Item_Discount"), ",", "").cast(DoubleType()).alias("item_discount"),
    regexp_replace(c("Item_Voided"), ",", "").cast(DoubleType()).alias("item_voided"),
    trim(c("Report_Group")).alias("report_group"),
    trim(c("Tender_Type")).alias("tender_type"),
    regexp_replace(c("Total_Rounding"), ",", "").cast(DoubleType()).alias("total_rounding")
)

print("Schema después de conversión:")
df_clean.printSchema()

In [0]:
# Verificar que la conversión funcionó correctamente
print("=== Muestra de datos limpios ===")
df_clean.show(10, truncate=False)

# Verificar fechas inválidas (null después de conversión)
df_clean.createOrReplaceTempView("df_clean_view")
bad_dates = spark.sql("SELECT COUNT(*) as total FROM df_clean_view WHERE date IS NULL").collect()[0]['total']
print(f"\nFechas inválidas (null): {bad_dates:,}")

# Verificar rangos de fechas
date_range = spark.sql("SELECT MIN(date) as min_date, MAX(date) as max_date FROM df_clean_view WHERE date IS NOT NULL").collect()[0]
print(f"Rango de fechas: {date_range['min_date']} a {date_range['max_date']}")

# Estadísticas básicas de columnas numéricas
print("\n=== Estadísticas numéricas ===")
spark.sql("""
    SELECT 'total' as col, COUNT(*) as n, ROUND(AVG(total),2) as avg, MIN(total) as min_val, MAX(total) as max_val FROM df_clean_view
    UNION ALL
    SELECT 'tax', COUNT(*), ROUND(AVG(tax),2), MIN(tax), MAX(tax) FROM df_clean_view
    UNION ALL
    SELECT 'quantity', COUNT(*), ROUND(AVG(quantity),2), MIN(quantity), MAX(quantity) FROM df_clean_view
    UNION ALL
    SELECT 'menu_price', COUNT(*), ROUND(AVG(menu_price),2), MIN(menu_price), MAX(menu_price) FROM df_clean_view
""").show()

In [0]:
# Revisar valores únicos en columnas categóricas clave
print("=== Transaction Type ===")
df_clean.groupBy("transaction_type").count().orderBy("count", ascending=False).show(20, truncate=False)

print("\n=== Pop (canal) ===")
df_clean.groupBy("pop").count().orderBy("count", ascending=False).show(20, truncate=False)

print("\n=== Report Group (top 20) ===")
df_clean.groupBy("report_group").count().orderBy("count", ascending=False).show(20, truncate=False)

print("\n=== Tender Type ===")
df_clean.groupBy("tender_type").count().orderBy("count", ascending=False).show(20, truncate=False)

In [0]:
# Guardar la tabla limpia usando CTAS (más eficiente en serverless)
clean_table = "workspace.default.taco_bueno_clean_data"

print(f"Guardando tabla limpia: {clean_table}...")
spark.sql("""
CREATE OR REPLACE TABLE workspace.default.taco_bueno_pricing_clean AS
SELECT
    CAST(Store AS INT) AS store_id,
    TO_DATE(Date, 'M/d/yyyy') AS date,
    CAST(Total AS DOUBLE) AS total,
    Check_Number AS check_number,
    Time_Card AS time_card,
    Begin_Time AS begin_time,
    End_Time AS end_time,
    CAST(Total_Discount AS DOUBLE) AS total_discount,
    CAST(Order_Voided AS DOUBLE) AS order_voided,
    CAST(Tax AS DOUBLE) AS tax,
    CAST(Quantity AS INT) AS quantity,
    TRIM(Transaction_Type) AS transaction_type,
    TRIM(Menu_Item) AS menu_item,
    Menu_Item_Ext_Ref AS menu_item_ext_ref,
    TRIM(Pop) AS pop,
    POS_Code AS pos_code,
    CAST(Menu_Price AS DOUBLE) AS menu_price,
    CAST(Item_Discount AS DOUBLE) AS item_discount,
    CAST(Item_Voided AS DOUBLE) AS item_voided,
    TRIM(Report_Group) AS report_group,
    TRIM(Tender_Type) AS tender_type,
    CAST(Total_Rounding AS DOUBLE) AS total_rounding
FROM workspace.default.taco_bueno_raw_data
""")

# Verificar resultado
final_count = spark.sql(f"SELECT COUNT(*) as total FROM {clean_table}").collect()[0]['total']
bad_dates = spark.sql(f"SELECT COUNT(*) as total FROM {clean_table} WHERE date IS NULL").collect()[0]['total']
print(f"\n✅ Tabla limpia creada: {clean_table}")
print(f"   Filas: {final_count:,}")
print(f"   Fechas nulas: {bad_dates:,}")
print(f"   Columnas: {len(df_clean.columns)}")

# Mostrar muestra final
print("\nMuestra final:")
spark.sql(f"SELECT store_id, date, total, quantity, transaction_type, menu_item, pop, menu_price FROM {clean_table} LIMIT 10").show(truncate=False)